In [ ]:
# ==========================================================
# SEMANA 4 - DATA TRADERS
# SCRAPING DINÁMICO + GUARDADO EN MONGODB
# ==========================================================

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from pymongo import MongoClient
import time

# ==========================================================
# 1. CONFIGURACIÓN GENERAL DEL PROYECTO
# ==========================================================
NOMBRE_GRUPO = "Data Traders"

urls = {
    "gainers": "https://finance.yahoo.com/markets/stocks/gainers/",
    "losers": "https://finance.yahoo.com/markets/stocks/losers/",
    "most_active": "https://finance.yahoo.com/markets/stocks/most-active/"
}

# ==========================================================
# 2. CONFIGURACIÓN DEL NAVEGADOR
# ==========================================================
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)

# ==========================================================
# 3. SCRAPING DINÁMICO
# ==========================================================
datos_finales = []

try:
    for categoria, url in urls.items():
        print(f"\n--- Procesando categoría: {categoria} ---")
        driver.get(url)
        time.sleep(5)

        filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
        print(f"Filas encontradas en {categoria}: {len(filas)}")

        for fila in filas:
            try:
                columnas = fila.find_elements(By.TAG_NAME, "td")

                if len(columnas) >= 7:
                    item_extraido = {
                        "categoria": categoria,
                        "simbolo": columnas[0].text,
                        "nombre": columnas[1].text,
                        "precio": columnas[2].text,
                        "cambio": columnas[3].text,
                        "porcentaje": columnas[4].text,
                        "volumen": columnas[6].text,
                        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "grupo": NOMBRE_GRUPO
                    }
                    datos_finales.append(item_extraido)

            except Exception as e:
                print("Error en una fila:", e)
                continue

    print(f"\nExtracción finalizada. Registros capturados: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en el scraping: {e}")

finally:
    driver.quit()
    print("Navegador cerrado.")

# ==========================================================
# 4. CONEXIÓN A MONGODB
# ==========================================================
try:
    # Según la guía, la conexión debe hacerse al servicio 'mongodb'
    client = MongoClient("mongodb://mongodb:27017/")
    db = client["proyecto_bigdata"]
    coleccion = db["datos_scraping"]
    print("CONEXIÓN ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXIÓN:", e)

# ==========================================================
# 5. INSERCIÓN DE DATOS EN MONGODB
# ==========================================================
registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        # Limpieza básica del precio
        precio_sucio = str(dato.get("precio", "0"))
        precio_limpio = (
            precio_sucio
            .replace("$", "")
            .replace(",", "")
            .strip()
        )

        try:
            dato["precio"] = float(precio_limpio)
        except ValueError:
            # Si no se puede convertir, se guarda como texto
            dato["precio"] = precio_sucio

        coleccion.insert_one(dato)
        registros_exitosos += 1

    except Exception as e:
        print("ERROR EN INSERCIÓN:", e)
        registros_fallidos += 1

# ==========================================================
# 6. RESUMEN DE CARGA
# ==========================================================
print("\nRESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

# ==========================================================
# 7. VERIFICACIÓN DE DATOS GUARDADOS
# ==========================================================
print("\nPrimeros documentos guardados en MongoDB:")
for doc in coleccion.find().limit(5):
    print(doc)